# A checked RNA-seq analysis input

This notebook exports the compact `rnaseq-005` case and inspects count quality and its paired design. The dataset is a published prostate tumour versus adjacent non-tumour comparison. The adjacent tissue is not a healthy control. Install `openomicsbench` before running the notebook.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from omicsbench import Collection

collection = Collection()
with TemporaryDirectory() as temporary:
    pack = Path(temporary) / 'inputs'
    collection.pack_rnaseq(pack, ['rnaseq-005'])
    case = pack / 'rnaseq-005'
    qc = collection.qc_rnaseq(case / 'counts.tsv', case / 'samples.tsv',
                              case / 'contrasts.tsv', 'rnaseq-005')
    print('Genes:', qc['features'], 'Samples:', qc['samples'])
    print('Design:', qc['design_check']['summary']['status'])
    print('Count QC:', qc['summary']['status'])
    for sample in qc['sample_qc'][:3]:
        print(sample['sample'], sample['library_size'], sample['detected_features'])
    assert qc['features'] == 8000 and qc['samples'] == 28
    assert qc['summary']['status'] == 'pass'


The check reports structural consistency and descriptive count metrics. It does not perform differential expression or infer sequencing quality from raw reads. Run your chosen statistical method with its documented input requirements and preserve the declared target-minus-reference contrast. The next cell assesses a native result table if you place it at `results.csv`.

In [ ]:
result_path = Path('results.csv')  # Replace with your DESeq2, edgeR or limma result file.
if result_path.is_file():
    report = collection.get('rnaseq-005').assess_de(result_path, alpha=0.05)
    print('Tested genes:', report['tested_rows'])
    print('Adjusted-p calls:', report['significant_rows'])
    print('Effect-direction agreement:', report['effect_direction_agreement'])
    print('Effect-rank Spearman:', report['effect_rank_spearman'])
    truth_path = Path('labels.tsv')  # Optional independent gene_id,is_de labels.
    if truth_path.is_file():
        truth_report = collection.get('rnaseq-005').assess_de(
            result_path, alpha=0.05, truth=truth_path)['independent_truth']
        print('Precision, recall, F1:', truth_report['precision'],
              truth_report['recall'], truth_report['f1'])
else:
    print('Add your native result table at results.csv to run the result check.')


Rank and direction agreement compare with one archived DESeq2 analysis. Only independently established labels supplied in `labels.tsv` support the precision, recall and F1 calculation within their labelled gene set. Neither comparison estimates false-discovery calibration.